# Chapter 22 - Interpretability, Made Visible (live)

Companion notebook for [Chapter 22](../part-6-frontier/22-interpretability.md). The chapter's most important ideas are inherently visual, so let's *see* them on tiny self-contained models:

1. **Linear probe** - is a concept linearly decodable from activations?
2. **Activation steering** - push behavior by adding a direction.
3. **Superposition** - more features than dimensions, packed as a pentagon.
4. **Sparse autoencoder (SAE)** - un-mix superposition into monosemantic features.

> Pure NumPy + matplotlib. Manual backprop throughout.

In [ ]:
class Adam:
    """Minimal Adam over a dict of numpy arrays (updated in place)."""
    def __init__(self, params, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
        self.p, self.lr, self.b1, self.b2, self.eps = params, lr, b1, b2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0
    def step(self, grads):
        self.t += 1
        for k in self.p:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            mh = self.m[k] / (1 - self.b1 ** self.t)
            vh = self.v[k] / (1 - self.b2 ** self.t)
            self.p[k] -= self.lr * mh / (np.sqrt(vh) + self.eps)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 1. Linear probe: is the concept decodable?

We synthesize activations that encode a binary concept along a hidden direction `u`, then train a linear probe. If it reaches high accuracy *and* its weight vector aligns with `u`, the concept is linearly represented - the basis of probing real models layer by layer.

In [ ]:
d = 32
u = np.random.randn(d); u /= np.linalg.norm(u)         # the true concept direction
N = 600
labels = np.random.randint(0, 2, N)
acts = (2 * labels - 1)[:, None] * 1.2 * u + np.random.randn(N, d)   # concept + noise

w = np.zeros(d); b = 0.0                                 # logistic probe via gradient descent
for _ in range(400):
    z = acts @ w + b; p = 1 / (1 + np.exp(-z))
    gw = acts.T @ (p - labels) / N; gb = (p - labels).mean()
    w -= 0.5 * gw; b -= 0.5 * gb

acc = (((acts @ w + b) > 0).astype(int) == labels).mean()
cos = abs(w @ u) / (np.linalg.norm(w) * np.linalg.norm(u))
print('probe accuracy: %.3f' % acc)
print('cosine(probe weight, true direction): %.3f' % cos)

## 2. Activation steering: move behavior along a direction

Take a class-0 activation and add $\alpha\,u$. As $\alpha$ grows, the probe's output crosses 0.5 and flips to class 1 - the same trick behind Golden Gate Claude and refusal-direction ablation: high-level behavior lives along a low-dimensional direction.

In [ ]:
x0 = acts[labels == 0][0]
alphas = np.linspace(0, 5, 60)
probs = [1 / (1 + np.exp(-((x0 + a * u) @ w + b))) for a in alphas]

plt.figure(figsize=(5, 3.3))
plt.plot(alphas, probs); plt.axhline(0.5, ls='--', color='gray')
plt.xlabel('steering strength a'); plt.ylabel('P(class 1)')
plt.title('Adding a*u steers the prediction across the boundary')
plt.tight_layout(); plt.show()

## 3. Superposition: 5 features in 2 dimensions

Anthropic's toy model: a network with only 2 hidden units must represent 5 sparse features. When features are **sparse**, it packs them as 5 near-equal-angle directions (a pentagon) - more features than dimensions. When features are **dense**, interference is too costly and it keeps only ~2.

In [ ]:
def train_toy(active_prob, n=5, m=2, steps=4000, seed=0):
    rng = np.random.default_rng(seed)
    p = {'W': rng.standard_normal((m, n)) * 0.1, 'b': np.zeros(n)}
    opt = Adam(p, lr=2e-2)
    for _ in range(steps):
        B = 1024
        active = (rng.random((B, n)) < active_prob)
        X = active * rng.random((B, n))                 # sparse non-negative features
        Hh = X @ p['W'].T                               # (B, m) bottleneck
        pre = Hh @ p['W'] + p['b']
        Xhat = np.maximum(0, pre)
        dpre = 2 * (Xhat - X) / B * (pre > 0)
        dHh = dpre @ p['W'].T                            # backprop into the bottleneck
        gW = Hh.T @ dpre + dHh.T @ X                     # W appears twice (encode + decode)
        gb = dpre.sum(0)
        opt.step({'W': gW, 'b': gb})
    return p['W']

fig, axes = plt.subplots(1, 2, figsize=(9, 4.3))
for ax, (name, ap) in zip(axes, [('dense (active_prob=1.0)', 1.0), ('sparse (active_prob=0.08)', 0.08)]):
    W = train_toy(ap)
    for i in range(W.shape[1]):
        ax.arrow(0, 0, W[0, i], W[1, i], head_width=0.04, length_includes_head=True, color='C0')
    lim = 1.2 * np.abs(W).max()
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(name)
plt.suptitle('Superposition: sparsity lets 5 features share 2 dimensions'); plt.tight_layout(); plt.show()

## 4. A sparse autoencoder un-mixes superposition

We synthesize activations as sparse combinations of hidden "true features," then train an over-complete SAE with an L1 penalty. Its decoder columns recover the true features - each SAE unit becomes **monosemantic**, the cure for polysemantic neurons.

In [ ]:
d, k = 16, 16
D = np.random.randn(d, k); D /= np.linalg.norm(D, axis=0, keepdims=True)   # true features
def gen(B):
    s = (np.random.rand(B, k) < 0.2) * np.random.rand(B, k)                # ~3 active of 16
    return s @ D.T

h = 64
sae = {'We': np.random.randn(d, h) * 0.1, 'be': np.zeros(h), 'Wd': np.random.randn(h, d) * 0.1}
opt = Adam(sae, lr=3e-3)
for _ in range(6000):
    X = gen(512)
    pre = X @ sae['We'] + sae['be']; F = np.maximum(0, pre)
    Xhat = F @ sae['Wd']
    dXhat = 2 * (Xhat - X) / 512
    gWd = F.T @ dXhat
    dF = dXhat @ sae['Wd'].T + 1e-2 * np.sign(F) / 512    # reconstruction + L1
    dpre = dF * (pre > 0)
    gWe = X.T @ dpre; gbe = dpre.sum(0)
    opt.step({'We': gWe, 'be': gbe, 'Wd': gWd})

dec = sae['Wd'].T                                          # (d, h) decoder directions
dec_n = dec / (np.linalg.norm(dec, axis=0, keepdims=True) + 1e-9)
sims = np.abs(D.T @ dec_n)                                 # (k true) x (h learned)
best = sims.max(1)
print('mean best-match cosine (true feature -> SAE feature): %.3f' % best.mean())

plt.figure(figsize=(6, 4))
plt.imshow(sims, aspect='auto', cmap='magma')
plt.xlabel('SAE feature'); plt.ylabel('true feature')
plt.title('Each true feature is recovered by some SAE unit')
plt.colorbar(fraction=0.046); plt.tight_layout(); plt.show()

## Takeaway

Concepts are often **linear directions**: you can *read* them (probes), *write* them (steering), watch networks **superpose** more of them than they have neurons, and **recover** them with sparse autoencoders. These toys are the exact mechanisms scaled up in real interpretability research - the difference is only size.

[Back to Chapter 22](../part-6-frontier/22-interpretability.md) | [Solutions](../solutions/22-interpretability-solutions.md)